In [ ]:
import duckdb

con = duckdb.connect("../../ProjectData.duckdb")
con.execute("CREATE SCHEMA IF NOT EXISTS gold")

In [ ]:
con.execute("""
    CREATE OR REPLACE VIEW silver_data AS
    SELECT * FROM read_parquet('../../data/silver/cleaned_data_load_date=2026-03-04.parquet')
""")

In [ ]:
con.execute("""
    SELECT COUNT(*)                       AS total,
           MIN(review_date)               AS earliest,
           MAX(review_date)               AS latest,
           COUNT(DISTINCT product_parent) AS unique_products
    FROM silver_data
""").df()

In [ ]:
con.execute("""
    SELECT
        product_parent,
        MIN(review_relative_rank) AS min_rank,
        MAX(review_relative_rank) AS max_rank,
        COUNT(*)                  AS product_review_count_check
    FROM (
        SELECT *,
            ROW_NUMBER() OVER (
                PARTITION BY product_parent ORDER BY review_date
            ) AS review_relative_rank
        FROM silver_data
    )
    GROUP BY product_parent
    ORDER BY product_review_count_check DESC
    LIMIT 10
""").df()

In [ ]:
con.execute("""
    WITH base AS (
        SELECT *,
            MAX(review_date) OVER ()                            AS dataset_max_date,
            ROW_NUMBER() OVER (
                PARTITION BY product_parent ORDER BY review_date
            )                                                   AS review_relative_rank,
            COUNT(*) OVER (PARTITION BY product_parent)         AS product_review_count,
            MIN(review_date) OVER (PARTITION BY product_parent) AS first_review_date,
            COUNT(*) OVER (
                PARTITION BY product_parent, review_date
            )                                                   AS reviews_same_day
        FROM silver_data
    )
    SELECT
        product_parent,
        review_date,
        review_relative_rank,
        product_review_count,
        DATE_DIFF('day', review_date, dataset_max_date)     AS review_age_days,
        DATE_DIFF('day', first_review_date, review_date)    AS days_since_first_review,
        MONTH(review_date)                                  AS review_month,
        DAYOFWEEK(review_date)                              AS review_dayofweek,
        CASE
            WHEN review_relative_rank <= CEIL(product_review_count * 0.10)
            THEN TRUE ELSE FALSE
        END                                                 AS is_early_review
    FROM base
    LIMIT 20
""").df()

In [ ]:
con.execute("""
    WITH base AS (
        SELECT *,
            MAX(review_date) OVER ()                              AS dataset_max_date,
            ROW_NUMBER() OVER (
                PARTITION BY product_parent ORDER BY review_date
            )                                                     AS review_relative_rank,
            COUNT(*) OVER (PARTITION BY product_parent)           AS product_review_count,
            MIN(review_date) OVER (PARTITION BY product_parent)   AS first_review_date,
            COUNT(*) OVER (
                PARTITION BY product_parent, review_date
            )                                                     AS reviews_same_day
        FROM silver_data
    ),
    ranked AS (
        SELECT *,
            DATE_DIFF('day', review_date, dataset_max_date)       AS review_age_days,
            DATE_DIFF('day', first_review_date, review_date)      AS days_since_first_review,
            MONTH(review_date)                                    AS review_month,
            DAYOFWEEK(review_date)                                AS review_dayofweek,
            CASE
                WHEN review_relative_rank <= CEIL(product_review_count * 0.10)
                THEN TRUE ELSE FALSE
            END                                                   AS is_early_review
        FROM base
    )
    SELECT
        SUM(CASE WHEN review_age_days < 0 THEN 1 ELSE 0 END)         AS negative_age,
        SUM(CASE WHEN days_since_first_review < 0 THEN 1 ELSE 0 END)  AS negative_days_since_first,
        SUM(CASE WHEN is_early_review = TRUE THEN 1 ELSE 0 END)       AS early_review_count,
        AVG(reviews_same_day)                                          AS avg_burst_size,
        MAX(reviews_same_day)                                          AS max_burst_size
    FROM ranked
""").df()

In [ ]:
con.execute("DROP TABLE IF EXISTS gold.review_temporal_features")
con.execute("""
    CREATE TABLE gold.review_temporal_features AS
    WITH base AS (
        SELECT *,
            MAX(review_date) OVER ()                              AS dataset_max_date,
            ROW_NUMBER() OVER (
                PARTITION BY product_parent ORDER BY review_date
            )                                                     AS review_relative_rank,
            COUNT(*) OVER (PARTITION BY product_parent)           AS product_review_count,
            MIN(review_date) OVER (PARTITION BY product_parent)   AS first_review_date,
            MAX(review_date) OVER (PARTITION BY product_parent)   AS last_review_date,
            COUNT(*) OVER (
                PARTITION BY product_parent, review_date
            )                                                     AS reviews_same_day
        FROM silver_data
    ),
    ranked AS (
        SELECT *,
            DATE_DIFF('day', review_date, dataset_max_date)       AS review_age_days,
            DATE_DIFF('day', first_review_date, review_date)      AS days_since_first_review,
            MONTH(review_date)                                    AS review_month,
            DAYOFWEEK(review_date)                                AS review_dayofweek,
            CASE
                WHEN review_relative_rank <= CEIL(product_review_count * 0.10)
                THEN TRUE ELSE FALSE
            END                                                   AS is_early_review,

            -- Review velocity: average reviews per day across the product's full review period.
            -- High-velocity products have more competition; a review must work harder to stand out.
            -- NULL for products where all reviews share the same date (NULLIF avoids division by zero).
            product_review_count * 1.0
                / NULLIF(DATE_DIFF('day', first_review_date, last_review_date), 0)
                AS reviews_per_day

        FROM base
    )
    SELECT * FROM ranked
""")
print("gold.review_temporal_features created.")

In [ ]:
con.execute("""
    SELECT
        COUNT(*)                                          AS total_rows,
        SUM(CASE WHEN is_early_review THEN 1 END)        AS early_reviews,
        ROUND(AVG(review_age_days), 1)                   AS avg_age_days,
        ROUND(AVG(days_since_first_review), 1)           AS avg_days_since_first,
        MAX(product_review_count)                        AS max_reviews_per_product,
        SUM(CASE WHEN reviews_per_day IS NULL THEN 1 END) AS single_date_products,
        ROUND(AVG(reviews_per_day), 3)                   AS avg_reviews_per_day,
        ROUND(MAX(reviews_per_day), 3)                   AS max_reviews_per_day
    FROM gold.review_temporal_features
""").df()

In [ ]:
from pathlib import Path
from datetime import date

gold_dir = Path("../../data/gold")
gold_dir.mkdir(parents=True, exist_ok=True)

output_file = gold_dir / f"temporal_features_load_date={date.today().isoformat()}.parquet"
con.execute(f"""
    COPY (SELECT * FROM gold.review_temporal_features)
    TO '{output_file.as_posix()}' (FORMAT PARQUET)
""")
print(f"Saved: {output_file.resolve()}")

In [ ]:
con.close()